In [23]:
import sys
from pathlib import Path

# Move from notebooks/ -> ml/
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project Root:", PROJECT_ROOT)

Project Root: /home/rupali-jha/solar-flare-watch/ml


In [24]:
import os
print(Path.cwd())
print(PROJECT_ROOT)
print(os.path.exists(PROJECT_ROOT / "src"))

/home/rupali-jha/solar-flare-watch/ml/notebooks
/home/rupali-jha/solar-flare-watch/ml
True


In [25]:
from src.data_loader import *
from src.preprocessing import *
from src.feature_engineering import *
from src.alignment import align_all_sources

# Lightcurve
lightcurve_df = preprocess_lightcurve(
    load_lightcurve("czt1")
)

# Spectra
metadata_df, counts_matrix, stat_err_matrix, channels = load_spectra("czt1")

observation_start = lightcurve_df["DATETIME"].min()

metadata_df, counts_matrix = preprocess_spectra(
    metadata_df,
    counts_matrix,
    observation_start
)

# Housekeeping
hk_df = preprocess_housekeeping(
    load_housekeeping()
)


Loaded hk.fits


In [26]:
lightcurve_features = engineer_peak_features(
    engineer_lightcurve_features(lightcurve_df)
)

spectral_features = engineer_spectral_features(
    metadata_df,
    counts_matrix,
    channels
)

hk_features = engineer_housekeeping_features(
    hk_df
)

In [27]:
print(lightcurve_features.shape)
print(spectral_features.shape)
print(hk_features.shape)

(43188, 42)
(2158, 23)
(5610, 88)


In [28]:
print(len(lightcurve_features.columns))
print(len(spectral_features.columns))
print(len(hk_features.columns))

42
23
88


In [29]:
print("Lightcurve :", lightcurve_features["DATETIME"].dtype)
print("Spectra    :", spectral_features["DATETIME"].dtype)
print("Housekeeping:", hk_features["DATETIME"].dtype)

Lightcurve : datetime64[us]
Spectra    : datetime64[ns]
Housekeeping: datetime64[ns]


In [30]:
for df in [lightcurve_features, spectral_features, hk_features]:
    df["DATETIME"] = (
        pd.to_datetime(df["DATETIME"])
        .astype("datetime64[ns]")
    )

In [31]:
print(lightcurve_features["DATETIME"].dtype)
print(spectral_features["DATETIME"].dtype)
print(hk_features["DATETIME"].dtype)

datetime64[ns]
datetime64[ns]
datetime64[ns]


In [32]:
from src.alignment import align_all_sources

dataset = align_all_sources(
    lightcurve_features,
    spectral_features,
    hk_features,
)

In [34]:
print(dataset.shape)

print(dataset.isna().sum().sum())

print(dataset.duplicated().sum())

print(dataset["DATETIME"].is_monotonic_increasing)

(43188, 151)
176
0
True


In [36]:
missing = dataset.isna().sum()

missing = missing[missing > 0]

print(missing)

SPEC_NUM             8
ROWID                8
TSTART               8
TSTOP                8
EXPOSURE             8
MID_TIME             8
spec_total_counts    8
spec_mean_counts     8
spec_std_counts      8
spec_max             8
spec_min             8
peak_channel         8
spectral_centroid    8
spectral_spread      8
spectral_entropy     8
low_energy           8
mid_energy           8
high_energy          8
hardness_ratio1      8
hardness_ratio2      8
dominant_fraction    8
active_channels      8
dtype: int64


In [37]:
spectral_cols = [
    c for c in spectral_features.columns
    if c != "DATETIME"
]

dataset[spectral_cols] = (
    dataset[spectral_cols]
    .ffill()
    .bfill()
)

In [38]:
print(dataset.isna().sum().sum())

0


In [39]:
from pathlib import Path

SAVE_DIR = Path("../data/processed")

SAVE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

dataset.to_parquet(
    SAVE_DIR / "aligned_dataset.parquet",
    engine="pyarrow",
    index=False,
)

dataset.to_csv(
    SAVE_DIR / "aligned_dataset.csv",
    index=False,
)